# 07 - Python 独有特性

这些是 Java 里完全没有的语法糖，掌握后代码量能减少一半。

## 今日目标（20-30 分钟）

- 会用列表推导式替代简单的 for 循环
- 理解生成器和 yield
- 掌握 enumerate / zip 等常用内置函数
- 了解 walrus 运算符

完成标准：通过末尾打卡题

## 1. 列表推导式（List Comprehension）

Python 最标志性的语法，一行代替 for 循环。

In [ ]:
# 说明：演示基础循环写法，便于与你熟悉的 Java for 循环建立映射。

# Java:
# List<Integer> squares = new ArrayList<>();
# for (int i = 0; i < 10; i++) {
#     squares.add(i * i);
# }

# Python 传统写法
squares = []
for i in range(10):
    squares.append(i * i)
print(squares)

# 列表推导式：一行搞定
squares = [i * i for i in range(10)]
print(squares)


In [ ]:
# 说明：本段示例对应「1. 列表推导式（List Comprehension）」，建议先看注释再运行，重点观察输出与预期是否一致。

# 带条件过滤
# Java Stream: list.stream().filter(x -> x % 2 == 0).collect(Collectors.toList())

nums = [1, 2, 3, 4, 5, 6, 7, 8]
evens = [x for x in nums if x % 2 == 0]
print(evens)  # [2, 4, 6, 8]

# 带条件 + 转换
result = [x * 10 for x in nums if x > 3]
print(result)  # [40, 50, 60, 70, 80]


In [ ]:
# 说明：本段示例对应「1. 列表推导式（List Comprehension）」，建议先看注释再运行，重点观察输出与预期是否一致。

# 嵌套推导式（少用，复杂了就用普通 for 循环）

# 展开二维列表
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [x for row in matrix for x in row]
print(flat)  # [1, 2, 3, 4, 5, 6, 7, 8, 9]

# 读法：外层 for 在前，内层 for 在后
# 等价于：
# for row in matrix:
#     for x in row:
#         flat.append(x)

# ⚠️ 坑：嵌套推导式超过 2 层就很难读，可读性比简洁更重要
# 下面这行虽然能运行，但没人能一眼看懂：
# result = [f(x) for outer in data for mid in outer for x in mid if condition(x)]
#
# 原则：如果推导式需要注释解释，就换成普通 for 循环
# 列表推导式的价值在于"一眼能理解"，复杂了就失去这个价值了


## 2. 字典推导式 + 集合推导式

In [ ]:
# 说明：本段示例对应「2. 字典推导式 + 集合推导式」，建议先看注释再运行，重点观察输出与预期是否一致。

# 字典推导式
names = ["alice", "bob", "cindy"]
name_lengths = {name: len(name) for name in names}
print(name_lengths)  # {'alice': 5, 'bob': 3, 'cindy': 5}

# 反转 dict 的 key-value
original = {"a": 1, "b": 2, "c": 3}
flipped = {v: k for k, v in original.items()}
print(flipped)  # {1: 'a', 2: 'b', 3: 'c'}

# ⚠️ 坑：反转 dict 时，如果原始 value 有重复，后面的 key 会静默覆盖前面的——
# bad = {"a": 1, "b": 1, "c": 2}
# flipped = {v: k for k, v in bad.items()}
# print(flipped)  # {1: 'b', 2: 'c'}  ← "a" 丢了！1 先映射到 "a"，再被 "b" 覆盖
#
# 反转前先确认 value 唯一：
# assert len(set(original.values())) == len(original), "values 有重复，不能安全反转"


In [ ]:
# 说明：本段示例对应「2. 字典推导式 + 集合推导式」，建议先看注释再运行，重点观察输出与预期是否一致。

# 集合推导式（去重）
words = ["hello", "world", "hello", "python"]
unique_lengths = {len(w) for w in words}
print(unique_lengths)  # {5, 6}  长度自动去重


## 3. 生成器（Generator）

惰性求值，不一次性生成所有数据，按需产出。处理大数据时节省内存。

In [ ]:
# 说明：演示生成器表达式与列表推导式的核心差异（惰性 vs 立即计算）。

# 生成器表达式：把 [] 换成 ()
squares_list = [i * i for i in range(5)]   # 列表，立即生成所有数据
squares_gen = (i * i for i in range(5))     # 生成器，惰性求值

print(type(squares_list))  # <class 'list'>
print(type(squares_gen))   # <class 'generator'>

# 生成器只能遍历一次
for x in squares_gen:
    print(x, end=" ")
print()

# 再遍历就是空的
for x in squares_gen:
    print(x, end=" ")  # 不会输出任何东西
print("(empty)")

# ⚠️ 坑：生成器传给函数后就被消耗掉了，不会报错但结果是空的——
# gen = (x for x in range(5))
# total = sum(gen)      # gen 被消耗完
# items = list(gen)     # items = []，静默 bug，sum 已经把 gen 用光了
#
# 如果一个生成器需要被多次使用，先用 list() 转换：
# data = list(gen)
# total = sum(data)
# items = data  # 可以重复使用


In [ ]:
# 说明：演示生成器按需产出数据，处理大数据时更省内存。

# yield 关键字：让函数变成生成器
# Java 没有等价语法（Java 的 Iterator 要写一堆样板代码）

def count_up(n: int):
    """从 0 数到 n-1，每次 yield 一个值"""
    i = 0
    while i < n:
        yield i  # 暂停函数，返回 i，下次调用时从这里继续
        i += 1

for num in count_up(5):
    print(num, end=" ")
print()


In [ ]:
# 说明：演示用 yield 按行流式处理文件，避免一次性加载大文件。

# 实际场景：逐行读取大文件，不会一次性加载到内存

def read_lines(path: str):
    with open(path) as f:
        for line in f:
            yield line.strip()

# 用法：
# for line in read_lines("big_file.txt"):
#     process(line)

print("生成器适合：大文件处理、分页查询、流式 API 响应")


## 4. 切片语法深入

In [ ]:
# 说明：本段示例对应「4. 切片语法深入」，建议先看注释再运行，重点观察输出与预期是否一致。

# 02 讲了基础切片，这里补充几个高频用法

nums = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# 负索引：从末尾数
print(nums[-1])     # 9（最后一个）
print(nums[-3:])    # [7, 8, 9]（最后三个）

# 步长
print(nums[::2])    # [0, 2, 4, 6, 8]（隔一个取一个）
print(nums[1::2])   # [1, 3, 5, 7, 9]（从索引1开始，隔一个取）

# 反转
print(nums[::-1])   # [9, 8, 7, 6, 5, 4, 3, 2, 1, 0]

# 字符串也支持切片（Java 要用 substring）
s = "Hello World"
print(s[:5])    # Hello
print(s[::-1])  # dlroW olleH


## 5. 常用内置函数

In [ ]:
# 说明：演示 enumerate 同时获取索引和值，避免手写 range(len(...))。

# enumerate：遍历时同时获取索引
# Java: for (int i = 0; i < list.size(); i++)

fruits = ["apple", "banana", "cherry"]

# 不用 enumerate 的写法（Java 风格）
for i in range(len(fruits)):
    print(i, fruits[i])

print("---")

# Pythonic 写法
for i, fruit in enumerate(fruits):
    print(i, fruit)

print("---")

# 指定起始索引
for i, fruit in enumerate(fruits, start=1):
    print(i, fruit)


In [ ]:
# 说明：演示 zip 并行遍历多个序列，常用于名称与分数等配对数据处理。

# zip：同时遍历多个列表（Java 没有直接等价物）

names = ["alice", "bob", "cindy"]
scores = [88, 95, 91]

for name, score in zip(names, scores):
    print(f"{name}: {score}")

# zip 成 dict
score_map = dict(zip(names, scores))
print(score_map)  # {'alice': 88, 'bob': 95, 'cindy': 91}

# 注意：zip 遇到长度不一致会按最短列表截断，超出的元素会被静默忽略

# 💡 实战：如果需要按最长列表遍历（短的用默认值补齐），用 itertools.zip_longest：
from itertools import zip_longest

names2 = ["alice", "bob", "cindy"]
scores2 = [88, 95]  # 少一个

for name, score in zip_longest(names2, scores2, fillvalue=0):
    print(f"{name}: {score}")
# 输出：alice: 88 / bob: 95 / cindy: 0  ← cindy 没有成绩，用 0 补齐
# 处理不完整数据（如分批 API 返回、可选字段对齐）时很有用


In [ ]:
# 说明：本段示例对应「5. 常用内置函数」，建议先看注释再运行，重点观察输出与预期是否一致。

# any / all：Java 的 Stream.anyMatch / allMatch

nums = [2, 4, 6, 8]
print(all(x % 2 == 0 for x in nums))  # True，全部是偶数
print(any(x > 5 for x in nums))       # True，存在大于5的

# 注意：参数是生成器表达式，不需要先生成完整列表


In [ ]:
# 说明：本段示例对应「5. 常用内置函数」，建议先看注释再运行，重点观察输出与预期是否一致。

# sorted / min / max 配合 key 参数

users = ["cindy", "bob", "alice"]
print(sorted(users))                    # 默认字母排序
print(sorted(users, key=len))           # 按长度排序
print(sorted(users, key=len, reverse=True))  # 按长度倒序

print(min(users, key=len))  # bob（最短）
print(max(users, key=len))  # cindy（最长）


## 6. 其他语法糖

In [ ]:
# 说明：演示解包与交换赋值，减少临时变量，让代码更简洁。

# 多重赋值与交换（02 里提过，这里补充更多场景）

# 同时赋值多个变量
x, y, z = 1, 2, 3
print(x, y, z)

# 连续赋值
a = b = c = 0
print(a, b, c)

# 交换
a, b = 10, 20
a, b = b, a
print(a, b)  # 20 10


In [ ]:
# 说明：本段示例对应「6. 其他语法糖」，建议先看注释再运行，重点观察输出与预期是否一致。

# 字符串乘法 + 列表乘法

print("ha" * 3)       # hahaha
print("-" * 40)       # 分隔线
print([0] * 5)        # [0, 0, 0, 0, 0]（快速初始化列表）


In [ ]:
# 说明：本段示例对应「6. 其他语法糖」，建议先看注释再运行，重点观察输出与预期是否一致。

# 链式比较（Java 不支持）

age = 25
# Java: age >= 18 && age < 60
print(18 <= age < 60)   # True
print(0 < age < 10)     # False


In [ ]:
# 说明：本段示例对应「6. 其他语法糖」，建议先看注释再运行，重点观察输出与预期是否一致。

# walrus 运算符 :=（Python 3.8+）
# 在表达式中赋值，减少重复计算

# 没有 walrus：
line = "hello world"
if len(line) > 5:
    print(f"long line ({len(line)} chars)")  # len 算了两次

# 有 walrus：
if (n := len(line)) > 5:
    print(f"long line ({n} chars)")  # len 只算一次，n 可以复用

# 常见场景：while 循环读取文件
# while (chunk := file.read(1024)):
#     process(chunk)

# 💡 实战：处理 AI 流式响应（Streaming API）时 walrus 很自然——
# OpenAI / Claude 的流式接口按块返回数据，用 walrus 逐块处理：
#
# import httpx
# with httpx.stream("POST", url, json=payload) as response:
#     while (chunk := response.read(1024)):
#         text = chunk.decode("utf-8")
#         process_chunk(text)
#
# 或者用 for 循环的生成器形式（更 Pythonic）：
# for chunk in response.iter_text():
#     process_chunk(chunk)
# walrus 在这里让"读取 + 判断是否结束"合并成一个表达式，避免读两次


## 7. Java 对照速记

| Java | Python |
|------|--------|
| `stream().map().filter().collect()` | 列表推导式 `[x for x in list if ...]` |
| `Iterator` 接口 | `yield`（生成器） |
| `for (int i = 0; i < n; i++)` | `for i, x in enumerate(list)` |
| 无 | `zip(list1, list2)` 并行遍历 |
| `stream().anyMatch()` | `any()` |
| `stream().allMatch()` | `all()` |
| `a >= 18 && a < 60` | `18 <= a < 60` |
| 无 | `:=` walrus 运算符 |
| 无 | `"ha" * 3` 字符串乘法 |

## 今日打卡题

1. 用列表推导式：给定 `words = ["hello", "hi", "hey", "howdy", "hola"]`，筛选出长度大于 3 的单词并转大写
2. 用字典推导式：给定 `keys = ["a", "b", "c"]` 和 `values = [1, 2, 3]`，生成 `{"a": 1, "b": 2, "c": 3}`
3. 写一个生成器函数 `fibonacci(n)`，产出前 n 个斐波那契数
4. 用 `enumerate` + `zip`：给定 `names` 和 `scores` 两个列表，输出 `"1. alice: 88"` 这样的格式

In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

# TODO: 先自己写


In [ ]:
# 说明：这是打卡题参考实现，建议先自己做一遍，再用它核对思路和语法。

# 参考答案（先自己写完再看）

# 1
words = ["hello", "hi", "hey", "howdy", "hola"]
result = [w.upper() for w in words if len(w) > 3]
print(result)  # ['HELLO', 'HOWDY', 'HOLA']

# 2
keys = ["a", "b", "c"]
values = [1, 2, 3]
d = {k: v for k, v in zip(keys, values)}
print(d)  # {'a': 1, 'b': 2, 'c': 3}
# 其实 dict(zip(keys, values)) 更简洁，但这里练习字典推导式

# 3
def fibonacci(n: int):
    a, b = 0, 1
    for _ in range(n):  # _ 表示不用的循环变量
        yield a
        a, b = b, a + b

print(list(fibonacci(10)))  # [0, 1, 1, 2, 3, 5, 8, 13, 21, 34]

# 4
names = ["alice", "bob", "cindy"]
scores = [88, 95, 91]

for i, (name, score) in enumerate(zip(names, scores), start=1):
    print(f"{i}. {name}: {score}")


---

## 语法速学完成！

01-07 全部跑完，你已经具备了写 Python 项目的语法基础。

接下来进入 **08 - 工程化基础**，开始学习 pytest、logging、pyproject.toml 等实际项目需要的工具。